# Section 7: Split Design and Grouped Leakage Control
Develops strict Stratified Group partitioning bounding leakages securely without rescanning source images.


In [1]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

OUTPUT_ROOT = r"D:\GradProj\Skin Cancer Dataset\pipeline_output"
manifest_path = os.path.join(OUTPUT_ROOT, "manifests", "training_eligible_manifest_post_metadata.csv")

d_splits = os.path.join(OUTPUT_ROOT, "splits")
os.makedirs(d_splits, exist_ok=True)

## Build Grouping Policies


In [2]:
df = pd.read_csv(manifest_path)

expected_total = 20513
expected_counts = {"NV": 12736, "MEL": 4468, "BCC": 3309}

print("=== SECTION 7 INPUT DATASET VALIDATION ===")
print(f"Loaded rows: {len(df)} (Expected: {expected_total})")

class_counts = df["final_authoritative_label"].value_counts()
for cls, expected in expected_counts.items():
    print(f"{cls}: {class_counts.get(cls, 0)} (Expected: {expected})")

errors = []
if len(df) != expected_total:
    errors.append(f"Total row mismatch: expected {expected_total}, got {len(df)}")

for cls, expected in expected_counts.items():
    actual = class_counts.get(cls, 0)
    if actual != expected:
        errors.append(f"{cls} mismatch: expected {expected}, got {actual}")

if errors:
    raise ValueError(
        "Section 7 must run on the frozen post-metadata dataset state, but mismatches were found:\n- "
        + "\n- ".join(errors)
    )

def determine_grouping(row):
    if pd.notna(row.get("lesion_id")):
        return str(row["lesion_id"]), "lesion_id"
    if pd.notna(row.get("family_id")):
        return str(row["family_id"]), "family_id"
    if pd.notna(row.get("canonical_match_id")):
        return str(row["canonical_match_id"]), "canonical_match_id_fallback"
    return str(row["base_id_candidate"]), "base_id_candidate_fallback"

grouping = df.apply(determine_grouping, axis=1)
df["effective_split_group_id"] = [g[0] for g in grouping]
df["grouping_source"] = [g[1] for g in grouping]

print("\n=== GROUPING SOURCE DISTRIBUTION ===")
display(
    df["grouping_source"]
    .value_counts()
    .rename_axis("grouping_source")
    .reset_index(name="row_count")
)

# Build group table
group_df = (
    df.groupby("effective_split_group_id")
    .agg(
        final_authoritative_label=("final_authoritative_label", lambda x: x.mode().iloc[0]),
        grouping_source=("grouping_source", lambda x: x.mode().iloc[0]),
        row_count=("effective_split_group_id", "size"),
        lesion_id_non_missing=("lesion_id", lambda x: x.notna().any())
    )
    .reset_index()
)

# Validate that no effective group contains more than one class label
group_label_counts = (
    df.groupby("effective_split_group_id")["final_authoritative_label"]
    .nunique()
)
multi_class_groups = group_label_counts[group_label_counts > 1]

print("\n=== GROUP TABLE SUMMARY ===")
print(f"Unique effective groups: {len(group_df)}")
print(f"Groups with >1 class label: {len(multi_class_groups)}")

if len(multi_class_groups) > 0:
    raise ValueError(
        f"Found {len(multi_class_groups)} effective groups containing multiple class labels. "
        "This must be resolved before splitting."
    )

print("\n=== GROUP LABEL DISTRIBUTION ===")
display(
    group_df["final_authoritative_label"]
    .value_counts()
    .rename_axis("group_label")
    .reset_index(name="group_count")
)

=== SECTION 7 INPUT DATASET VALIDATION ===
Loaded rows: 20513 (Expected: 20513)
NV: 12736 (Expected: 12736)
MEL: 4468 (Expected: 4468)
BCC: 3309 (Expected: 3309)

=== GROUPING SOURCE DISTRIBUTION ===


,grouping_source,row_count
0,lesion_id,18779
1,family_id,1734



=== GROUP TABLE SUMMARY ===
Unique effective groups: 11469
Groups with >1 class label: 0

=== GROUP LABEL DISTRIBUTION ===


,group_label,group_count
0,NV,8524
1,MEL,1637
2,BCC,1308


## Perform Hierarchical Grouped Split


In [3]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f"Using split ratios: train={TRAIN_RATIO}, val={VAL_RATIO}, test={TEST_RATIO}")

# Step 1: split groups into train vs temp
group_train, group_temp = train_test_split(
    group_df,
    test_size=(1.0 - TRAIN_RATIO),
    stratify=group_df["final_authoritative_label"],
    random_state=42
)

# Step 2: split temp into val vs test
temp_ratio = VAL_RATIO + TEST_RATIO
val_relative = VAL_RATIO / temp_ratio

group_val, group_test = train_test_split(
    group_temp,
    test_size=(1.0 - val_relative),
    stratify=group_temp["final_authoritative_label"],
    random_state=42
)

group_train = group_train.copy()
group_val = group_val.copy()
group_test = group_test.copy()

group_train["split_assignment"] = "train"
group_val["split_assignment"] = "val"
group_test["split_assignment"] = "test"

group_assignments = pd.concat([group_train, group_val, group_test], axis=0)

# Map group split back to row-level dataframe
df_final = df.merge(
    group_assignments[["effective_split_group_id", "split_assignment"]],
    on="effective_split_group_id",
    how="left"
)

if df_final["split_assignment"].isna().any():
    raise ValueError("Some rows did not receive a split_assignment. This should not happen.")

df_train = df_final[df_final["split_assignment"] == "train"].copy()
df_val = df_final[df_final["split_assignment"] == "val"].copy()
df_test = df_final[df_final["split_assignment"] == "test"].copy()

print("\n=== SPLIT COUNTS ===")
print(f"Train rows: {len(df_train)}")
print(f"Val rows: {len(df_val)}")
print(f"Test rows: {len(df_test)}")

Using split ratios: train=0.7, val=0.15, test=0.15

=== SPLIT COUNTS ===
Train rows: 14416
Val rows: 3000
Test rows: 3097


In [4]:
print("\n=== SPLIT SUMMARY ===")
print("1. Split Row Counts:")
print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

print("\n2. Class Counts per Split:")
split_class_counts = pd.crosstab(df_final["split_assignment"], df_final["final_authoritative_label"])
display(split_class_counts)

# Effective group leakage check
group_splits = df_final.groupby("effective_split_group_id")["split_assignment"].nunique()
leaked_groups = group_splits[group_splits > 1]

print("\n3. Effective Group Leakage Check:")
print(f"Groups spanning multiple splits: {len(leaked_groups)}")

# lesion_id leakage check
lesion_rows = df_final[df_final["lesion_id"].notna()].copy()
lesion_split_counts = lesion_rows.groupby("lesion_id")["split_assignment"].nunique()
leaked_lesions = lesion_split_counts[lesion_split_counts > 1]

print("\n4. Lesion ID Leakage Check:")
print(f"Lesion IDs spanning multiple splits: {len(leaked_lesions)}")

# Total integrity check
sum_totals = len(df_final) == len(df)
print("\n5. Total Integrity Check:")
print(f"Split totals sum back to full dataset: {sum_totals} ({len(df_final)} rows)")

# Metadata support by split
lesion_id_coverage_by_split = (
    df_final.groupby("split_assignment")
    .agg(
        total_rows=("split_assignment", "size"),
        rows_with_lesion_id=("lesion_id", lambda x: x.notna().sum()),
        rows_without_lesion_id=("lesion_id", lambda x: x.isna().sum())
    )
    .reset_index()
)

print("\n6. Lesion ID Coverage by Split:")
display(lesion_id_coverage_by_split)

# Grouping source distribution by split
grouping_source_by_split = pd.crosstab(df_final["split_assignment"], df_final["grouping_source"])
print("\n7. Grouping Source Distribution by Split:")
display(grouping_source_by_split)

# Fail loudly if leakage exists
if len(leaked_groups) > 0:
    raise ValueError(f"Found {len(leaked_groups)} effective groups leaking across multiple splits.")

if len(leaked_lesions) > 0:
    raise ValueError(f"Found {len(leaked_lesions)} lesion_id values leaking across multiple splits.")

if not sum_totals:
    raise ValueError("Split totals do not sum back to the full dataset.")

# Save manifests
cols_to_save = [
    "full_path",
    "final_authoritative_label",
    "canonical_match_id",
    "lesion_id",
    "family_id",
    "effective_split_group_id",
    "grouping_source",
    "split_assignment"
]

df_train[cols_to_save].to_csv(os.path.join(d_splits, "train_manifest.csv"), index=False)
df_val[cols_to_save].to_csv(os.path.join(d_splits, "val_manifest.csv"), index=False)
df_test[cols_to_save].to_csv(os.path.join(d_splits, "test_manifest.csv"), index=False)

# Save reports
split_summary_report = pd.DataFrame([
    {"split": "train", "total_rows": len(df_train)},
    {"split": "val", "total_rows": len(df_val)},
    {"split": "test", "total_rows": len(df_test)}
])
split_summary_report.to_csv(os.path.join(d_splits, "split_summary_report.csv"), index=False)

grouping_source_by_split.to_csv(os.path.join(d_splits, "split_grouping_report.csv"))

integrity = pd.DataFrame([{
    "leaked_effective_groups": len(leaked_groups),
    "leaked_lesion_ids": len(leaked_lesions),
    "sum_matches_total": sum_totals
}])
integrity.to_csv(os.path.join(d_splits, "split_integrity_checks.csv"), index=False)

split_class_counts.to_csv(os.path.join(d_splits, "class_distribution_by_split.csv"))
lesion_id_coverage_by_split.to_csv(os.path.join(d_splits, "lesion_id_coverage_by_split.csv"), index=False)

print("\n=== SECTION 7 FINAL RULE ===")
print("All later preprocessing and training must use:")
print("- train_manifest.csv")
print("- val_manifest.csv")
print("- test_manifest.csv")
print("These are now the authoritative split sources.")


=== SPLIT SUMMARY ===
1. Split Row Counts:
Train: 14416 | Val: 3000 | Test: 3097

2. Class Counts per Split:


final_authoritative_label,BCC,MEL,NV
split_assignment,,,
test,486,672,1939
train,2333,3172,8911
val,490,624,1886



3. Effective Group Leakage Check:
Groups spanning multiple splits: 0

4. Lesion ID Leakage Check:
Lesion IDs spanning multiple splits: 0

5. Total Integrity Check:
Split totals sum back to full dataset: True (20513 rows)

6. Lesion ID Coverage by Split:


,split_assignment,total_rows,rows_with_lesion_id,rows_without_lesion_id
0,test,3097,2866,231
1,train,14416,13188,1228
2,val,3000,2725,275



7. Grouping Source Distribution by Split:


grouping_source,family_id,lesion_id
split_assignment,,
test,231,2866
train,1228,13188
val,275,2725



=== SECTION 7 FINAL RULE ===
All later preprocessing and training must use:
- train_manifest.csv
- val_manifest.csv
- test_manifest.csv
These are now the authoritative split sources.
